In [125]:
import pandas as pd
import numpy as np

In [126]:
class LikertSurvey:
    def __init__(self, n_questions, csv_path="USSF Likert Survey Questions.csv", randomize_responses=False, prompt_user=False):
        self.csv_path = csv_path
        self.randomize_responses = randomize_responses
        self.prompt_user = prompt_user
        self.n_questions = n_questions
        self.survey = []

        assert self.randomize_responses or self.prompt_user, "At least one of randomize_responses or prompt_user must be True"
        assert self.n_questions <= 100, "n_questions must be less than or equal to the number of questions in the survey"   

        self._load_questions()

    def _load_questions(self):
        try:
            df = pd.read_csv(self.csv_path)
        except FileNotFoundError:
            print(f"File not found: {self.csv_path}")
            return []
        
        survey = []
        for _, row in df.iterrows():
            question_tuple = (row['Statement'], None)
            if self.prompt_user:
                question_tuple = (row['Statement'], self._ask_question(row['Statement']))
            elif self.randomize_responses:
                question_tuple = (row['Statement'], np.random.randint(1, 6))
            
            if question_tuple[1] is not None:
                survey.append(question_tuple)
                if len(survey) >= self.n_questions:
                    break         
        self.survey = survey
        return self.survey


    def _ask_question(self, question):
        print(question)
        response = input("Enter your response 1-5 (1 being strongly disagree, 5 being strongly agree): ")
        if response not in ['1', '2', '3', '4', '5']:
            print("Please enter a number between 1 and 5.")
            return self.prompt_user(question)
        return int(response)

    def get_survey(self):
        return self.survey    

In [127]:
guardianRandomResponse = LikertSurvey(n_questions=15, randomize_responses=True)
guardianRandomResponse.get_survey()

[('My work contributes meaningfully to national security.', 3),
 ('I am satisfied with my current duty location.', 5),
 ('The USSF provides clear career progression paths.', 2),
 ("The USSF's technology and equipment are adequate for mission success.", 2),
 ('The USSF provides adequate recognition for achievements.', 3),
 ('I have adequate access to continuing education opportunities.', 3),
 ('My unit maintains high standards of performance.', 4),
 ('My supervisor provides helpful feedback on my performance.', 5),
 ('The USSF effectively manages workload distribution.', 2),
 ('The USSF supports my personal and family needs.', 4),
 ('The USSF provides clear guidance on professional conduct.', 1),
 ('The USSF provides clear expectations for performance.', 3),
 ('The USSF effectively communicates its achievements to the public.', 1),
 ('The USSF provides adequate financial planning resources.', 2),
 ('The USSF provides adequate opportunities for joint assignments.', 1)]

In [128]:
# Monte Carlo Simulation of 1000 Randomized Surveys with all 100 Questions 
from tqdm import tqdm

survey_list = []
for i in tqdm(range(1000)):
    guardian = LikertSurvey(n_questions=100, randomize_responses=True)
    survey = guardian.get_survey()
    survey_list.append(survey)

# Convert the list of surveys into a DataFrame
# Each unique question should be a column where each row is a sample
unique_questions = set(item[0] for sublist in survey_list for item in sublist)
df_dict = {question: [] for question in unique_questions}

for survey in survey_list:
    survey_dict = {question: None for question in unique_questions}
    for question, response in survey:
        survey_dict[question] = response
    for question in unique_questions:
        df_dict[question].append(survey_dict[question])

df = pd.DataFrame(df_dict)

100%|██████████| 1000/1000 [00:04<00:00, 221.01it/s]


In [129]:
df.head()

,I have adequate opportunities for operational experience.,The USSF offers adequate specialized training for my career field.,My compensation is competitive with civilian alternatives.,The USSF provides adequate opportunities for joint assignments.,I feel connected to the USSF's vision for the future.,I have adequate opportunities to provide feedback to leadership.,I feel the USSF is at the cutting edge of space operations.,I feel physically safe in my work environment.,I am satisfied with the USSF's leave policies.,The USSF provides adequate recreational facilities and programs.,...,The USSF provides adequate support during PCS moves.,The USSF provides adequate support for continuing education.,I am satisfied with my current duty location.,The USSF provides adequate technical training.,The USSF effectively addresses diversity and inclusion.,I have access to the resources needed to perform my job effectively.,The USSF effectively addresses burnout and stress management.,My chain of command is responsive to my concerns.,I am satisfied with the camaraderie within my unit.,I would recommend the USSF to qualified candidates.
0,3,3,2,4,5,1,1,2,1,2,...,2,5,2,5,3,1,5,3,5,5
1,4,5,4,3,2,2,3,2,1,1,...,5,4,5,5,3,4,3,4,4,3
2,2,3,5,1,5,1,5,1,1,2,...,5,4,2,3,2,5,5,3,1,2
3,2,5,5,4,1,5,3,3,4,1,...,4,5,4,1,5,5,3,1,4,3
4,2,3,1,4,3,1,1,3,2,1,...,2,3,5,2,2,2,5,4,4,5


In [130]:
# Transpose the DataFrame so that headers of the original df become rows in column 1
survey_analytics = df.describe().transpose().reset_index()
survey_analytics = survey_analytics.drop(columns=['count', 'min', 'max','25%','50%','75%'])
survey_analytics.head()

,index,mean,std
0,I have adequate opportunities for operational ...,3.011,1.423694
1,The USSF offers adequate specialized training ...,2.986,1.416266
2,My compensation is competitive with civilian a...,2.905,1.379086
3,The USSF provides adequate opportunities for j...,3.014,1.373933
4,I feel connected to the USSF's vision for the ...,3.044,1.417065


In [131]:
# pre-process the original df, map 1-5 to 0-1 decimal
df = df.applymap(lambda x: {1: 0, 2: 0.25, 3: 0.5, 4: 0.75, 5: 1}.get(x, x))
df.head()



,I have adequate opportunities for operational experience.,The USSF offers adequate specialized training for my career field.,My compensation is competitive with civilian alternatives.,The USSF provides adequate opportunities for joint assignments.,I feel connected to the USSF's vision for the future.,I have adequate opportunities to provide feedback to leadership.,I feel the USSF is at the cutting edge of space operations.,I feel physically safe in my work environment.,I am satisfied with the USSF's leave policies.,The USSF provides adequate recreational facilities and programs.,...,The USSF provides adequate support during PCS moves.,The USSF provides adequate support for continuing education.,I am satisfied with my current duty location.,The USSF provides adequate technical training.,The USSF effectively addresses diversity and inclusion.,I have access to the resources needed to perform my job effectively.,The USSF effectively addresses burnout and stress management.,My chain of command is responsive to my concerns.,I am satisfied with the camaraderie within my unit.,I would recommend the USSF to qualified candidates.
0,0.50,0.5,0.25,0.75,1.00,0.00,0.0,0.25,0.00,0.25,...,0.25,1.00,0.25,1.00,0.50,0.00,1.0,0.50,1.00,1.00
1,0.75,1.0,0.75,0.50,0.25,0.25,0.5,0.25,0.00,0.00,...,1.00,0.75,1.00,1.00,0.50,0.75,0.5,0.75,0.75,0.50
2,0.25,0.5,1.00,0.00,1.00,0.00,1.0,0.00,0.00,0.25,...,1.00,0.75,0.25,0.50,0.25,1.00,1.0,0.50,0.00,0.25
3,0.25,1.0,1.00,0.75,0.00,1.00,0.5,0.50,0.75,0.00,...,0.75,1.00,0.75,0.00,1.00,1.00,0.5,0.00,0.75,0.50
4,0.25,0.5,0.00,0.75,0.50,0.00,0.0,0.50,0.25,0.00,...,0.25,0.50,1.00,0.25,0.25,0.25,1.0,0.75,0.75,1.00


In [132]:
# Train a Logistic Regression Model 
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Generate a random target Variable
df['STATUS'] = np.random.choice([0, 1], size=len(df), p=[0.6, 0.4])

# Split the data into training and testing sets 
X_train, X_test, y_train, y_test = train_test_split(df, df['STATUS'], test_size=0.3, random_state=42)
X_train = X_train.drop(columns=['STATUS'])
X_test = X_test.drop(columns=['STATUS'])

# Train a Logistic Regression Model
model = LogisticRegression(max_iter=200)  # Increase max_iter to avoid convergence warnings

# Re-train the Logistic Regression Model with corrected features
model.fit(X_train, y_train)

# Make predictions on the test set with corrected features
y_pred = model.predict(X_test)

# Evaluate the model with corrected features
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred, zero_division=1)  # Handle zero division in classification report

# Print the corrected accuracy and report
print(accuracy)
print(report)

0.5333333333333333
              precision    recall  f1-score   support

           0       0.56      0.78      0.65       167
           1       0.45      0.23      0.30       133

    accuracy                           0.53       300
   macro avg       0.50      0.50      0.47       300
weighted avg       0.51      0.53      0.49       300



In [133]:
# Check if the implementation correctly identifies the top 10 most important features
feature_names = X_train.columns
coefficients = model.coef_[0]  

# Create a DataFrame to hold feature names and their corresponding coefficients
feature_importance = pd.DataFrame({
    'Question': feature_names,
    'Weight': coefficients
})

feature_importance = feature_importance.reindex(feature_importance.Weight.abs().sort_values(ascending=False).index)
print(feature_importance)

                                             Question    Weight
78    The USSF supports my personal and family needs. -0.888726
6   I feel the USSF is at the cutting edge of spac...  0.558894
19  I have adequate opportunities to network with ... -0.513320
13  My training has adequately prepared me for my ... -0.505985
64  I feel the USSF is respected by other military...  0.493963
..                                                ...       ...
61  The USSF provides adequate recognition for ach...  0.018485
41  I am satisfied with the USSF's organizational ... -0.012501
53  I am satisfied with healthcare services availa...  0.008107
86               The benefits package meets my needs.  0.007582
79        The USSF provides adequate housing support.  0.005697

[100 rows x 2 columns]


In [147]:
import ollama

def embed_text_local(text):
    response = ollama.embed(
        model="nomic-embed-text",
        input=text,
    )
    return response.embeddings[0]

feature_importance['embedding'] = None  
for index, row in tqdm(feature_importance.iterrows(), total=feature_importance.shape[0]):
    embedding_vector = embed_text_local(row['Question'])
    feature_importance.at[index, 'embedding'] = embedding_vector

100%|██████████| 100/100 [00:03<00:00, 31.63it/s]


In [162]:
# apply clustering to the embeddings to find 5 clusters
from sklearn.cluster import KMeans

# Initialize KMeans with 5 clusters
kmeans = KMeans(n_clusters=5, random_state=42)

# Fit the KMeans model to the embeddings
kmeans.fit(np.vstack(feature_importance['embedding'].values))

# Add the cluster labels to the feature_importance DataFrame
feature_importance['cluster'] = kmeans.labels_

# For each Cluster Print the top 5 Questions
for cluster in range(5):
    print(f"Group {cluster+1}:")
    top_questions = feature_importance[feature_importance['cluster'] == cluster].sort_values(by='Weight', ascending=False)
    for _, row in top_questions.iterrows():
        if row['Weight'] > 0:
            print(f"\t {row['Weight']:.2f} \t {row['Question']} ")
        else:
            print(f"\t{row['Weight']:.2f} \t {row['Question']} ")
    print("\n")



Group 1:
	 0.22 	 My supervisor provides helpful feedback on my performance. 
	 0.12 	 My work environment is positive and supportive. 
	-0.12 	 My chain of command is responsive to my concerns. 
	-0.12 	 I have adequate opportunities to provide feedback to leadership. 
	-0.21 	 I have confidence in my peers' abilities. 


Group 2:
	 0.56 	 I feel the USSF is at the cutting edge of space operations. 
	 0.49 	 I feel the USSF is respected by other military branches. 
	 0.40 	 The USSF culture aligns with my personal values. 
	 0.37 	 I feel the USSF is innovative in its approach to space operations. 
	 0.34 	 I am satisfied with the USSF uniform and appearance standards. 
	 0.26 	 I feel my career field is valued within the USSF. 
	 0.23 	 I am satisfied with the quality of USSF leadership training. 
	 0.22 	 I understand the USSF's promotion criteria. 
	 0.17 	 I would recommend the USSF to qualified candidates. 
	 0.16 	 The USSF offers adequate specialized training for my career fiel